#  XAITK-Saliency

The `xaitk-saliency` package is an open source, Explainable AI (XAI) framework
for visual saliency algorithm interfaces and implementations, built for
analytics and autonomy applications.

See [here](https://xaitk-saliency.readthedocs.io/en/latest/introduction.html)
for a more formal introduction to the topic of XAI and visual saliency
explanations.

This framework is a part of the [Explainable AI Toolkit (XAITK)](
https://xaitk.org).


## Explanainable AI (XAI)

Perturbation-occlusion methods are a technique used in computer vision to understand what parts of an image are important for a model's predictions.
RISE (*Randomized Input Sampling for Explanation*) and DRISE (*Dynamic RISE*) are two specific algorithms that generate visual saliency maps (*'intuitive heatmaps'*) that
highlight which regions of the image are most relevant to the decision-making process of a neural network.

(*See also the 'Terminology' section at the bottom of this notebook for definitions.*)

## The basic idea 

**If removing or modifying a part of the image changes the prediction significantly, that part is deemed important.**

## A picture tells a thousand words

What is RISE? Well, it's:

<!-- Replace with ![Alt text](rise.png) if using a notebook -->

![](https://eclique.github.io/rep-imgs/RISE/rise-overview.png)

*Image taken from [https://github.com/eclique/RISE/tree/master](https://github.com/eclique/RISE/tree/master)*

## What's happening?

1. Random binary masks are generated (where 1 keeps the pixel and 0 occludes it).
2. The image is multiplied by each mask, creating multiple perturbed images.
3. Each perturbed image is fed into the model, and the output scores (e.g., probabilities for different classes) are recorded.
4. These results are aggregated to form a saliency map—a heatmap showing which areas were most critical for the prediction.

## Let's see some code!

In [ ]:
import sys
import os

# The dataset of images is bundled inside /tests. 
# We do some path hacking to get access in this notebook.
sys.path.insert(0, os.path.abspath("../../tests"))

In [ ]:
from jatic_ri.object_detection.models import TorchvisionODModel
from jatic_ri.object_detection.datasets import CocoDetectionDataset

dataset = CocoDetectionDataset(
    root='../../tests/testing_utilities/example_data/coco_dataset',
    ann_file='../../tests/testing_utilities/example_data/coco_dataset/ann_file.json',
)
dataset = [dataset[1]]  # just select one image for demonstration purposes

model = TorchvisionODModel(model_name="fasterrcnn_resnet50_fpn_v2", device="cpu")

In [ ]:
# there are multiple "N/A" labels in the torchvision label mappings.
# xaitk-saliency currently de-duplicates, and this raises an error later in the pipeline.
# see https://gitlab.jatic.net/jatic/kitware/xaitk-jatic/-/issues/247
# here we add a unique suffix to all "N/A" labels to make sure there are no duplicates.

suffix=0
for val in model.index2label:
    if model.index2label[val] == "N/A":
        model.index2label[val] = f"N/A {suffix}"
        suffix+=1

In [ ]:
from jatic_ri.object_detection.test_stages import XAITKTestStage

# Important: n=200 will require considerable compute (e.g 16 vCPUs) and time (e.g. 15 minutes)
# to run. Try reducing to a smaller number initially (e.g. 5) to get faster results.
initialize_args = {
    "name": "XAITKTestStage Example", # label for stage, used in cache file
    "saliency_generator": {
        "type": "xaitk_saliency.impls.gen_object_detector_blackbox_sal.drise.DRISEStack",
        "xaitk_saliency.impls.gen_object_detector_blackbox_sal.drise.DRISEStack": {
            "n": 200, # Number of random masks used in the algorithm, larger means more compute
            "s": 8, # Spatial resolution of the small masking grid
            "p1": 0.5, # Probability of the grid cell being set to 1 (not occluded).
            "seed": 0, # A seed to pass to the constructed random number generator to allow for reproducibility.
            "threads": 4, # The number of threads to utilize when generating masks.
        },
    }, 
}
test_stage = XAITKTestStage(initialize_args)

In [ ]:
from jatic_ri.object_detection.metrics import map50_torch_metric_factory

test_stage.load_model(model=model, model_id="fasterrcnn_resnet50_fpn_v2")
test_stage.load_dataset(dataset=dataset, dataset_id="coco")
test_stage.load_metric(metric=map50_torch_metric_factory(), metric_id="mAP_50")

In [ ]:
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

test_stage.run(use_stage_cache=True)

In [ ]:
from gradient.templates_and_layouts.create_deck import create_deck
from pathlib import Path

gradient_slides = test_stage.collect_report_consumables()

# creation of powerpoint slide summarizing the results of the analysis
deck = create_deck(gradient_slides, path=Path("artifacts"), deck_name="XAITKDeck")

## What does a saliency map look like?

In [ ]:
from PIL import Image

# WARNING: these algorithms are stochastic, and so the result will be machine
# dependent. the path will also differ depending on where your cache is
# created.
Image.open("../../assets/saliency_map.png")

## Terminology
- *Perturbation*: Small modifications or masking of an image are introduced systematically or randomly to measure their effect on the model’s prediction.
- *Occlusion*: Parts of the image are obscured or masked with the goal of observing how much a prediction changes when different regions are altered.
- *Saliency Map*: A visual representation (often in the form of a heatmap) that indicates the most influential regions in an image for a given prediction. Saliency maps allow users to interpret which areas of an input are prioritized by the model, providing insights into the "why" behind a prediction.
- *RISE*: Algorithm that generates saliency maps by using random masks applied to the input image and measuring how much each masked version affects the model’s output.
- *DRISE*: Improves RISE by using adaptive masks based on the prediction at runtime. Instead of pre-computed random masks, DRISE generates masks dynamically, adjusting them according to the intermediate model outputs. This dynamic approach allows for more targeted exploration of the image space, focusing on regions that are more likely to be relevant.